# Show a GNoME structure (nglview)

Change `FORMULA`, Run All, drag to rotate.

**Kernel must be `structview (nglview)`** — check the top-right of the window. This notebook
does not work on the `ml_env` kernel: that environment has nglview 3.1.4, which does not
render on JupyterLab 4. `structview` has nglview 4.0.1, which does.

Launch Lab from this environment too, not just the kernel:

```
"/Users/mac/miniconda3/envs/structview/bin/jupyter-lab"
```

In [ ]:
FORMULA = "Ba2Sr6Sb4H2O"

In [ ]:
import os, zipfile, warnings
import pandas as pd
warnings.filterwarnings("ignore")
from pymatgen.core import Composition, Structure

ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()

# 168 MB - loaded once per session. dtype=str because 34,046 GNoME ids
# begin with a leading zero and would lose it to number parsing.
if "SUMMARY" not in globals():
    SUMMARY = pd.read_csv(os.path.join(ROOT, "gnome_data", "stable_materials_summary.csv"),
                          dtype={"MaterialId": str})

target = Composition(FORMULA).reduced_formula
hits = SUMMARY[SUMMARY["Reduced Formula"].map(
    lambda f: pd.notna(f) and Composition(str(f)).reduced_formula == target)]

if hits.empty:
    raise SystemExit(f"{FORMULA} is not in GNoME")

row = hits.iloc[0]
z = zipfile.ZipFile(os.path.join(ROOT, "gnome_data", "by_composition.zip"))
cif_text = z.read(f"by_composition/{row['Composition']}.CIF").decode()
st = Structure.from_str(cif_text, fmt="cif")

print(f"{row['Reduced Formula']}   {row['MaterialId']}   {row['Space Group']}   {len(st)} sites")
if len(hits) > 1:
    print(f"({len(hits)} matches - showing the first; use row = hits.iloc[n] for the others)")

In [ ]:
import nglview

# show_pymatgen goes through pymatgen's ASE adaptor, so `ase` must be installed
# or it hands nglview an MSONAtoms with no .write and dies. It is installed here.
view = nglview.show_pymatgen(st)
view.add_unitcell()
view.add_spacefill(radius=0.6)
view.center()
view

Useful tweaks:

```python
view.add_ball_and_stick()        # instead of spacefill
view.remove_spacefill()
view.background = "black"
view.camera = "orthographic"     # better for crystals than the default perspective
view.download_image("structure.png")
```

If the viewer is blank, the kernel is almost certainly `ml_env` rather than `structview`.